## Task 3: ML Model Portfolio

In [16]:
# !pip install optuna
from pyspark.sql import SparkSession
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    DecisionTreeClassifier,
    NaiveBayes
)
import time
import json
import pandas as pd
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
import optuna


In [44]:
spark_session = (
    SparkSession.builder
    .appName("Task_3_session")
    .config("spark.driver.memory", "6g")
    .config("spark.executor.memory", "6g")
    .getOrCreate()
)


In [45]:
training_dataframe = spark_session.read.parquet(
    "data/processed/training_dataset"
)

testing_dataframe = spark_session.read.parquet(
    "data/processed/testing_dataset"
)

In [4]:
print(f"Training Records : {training_dataframe.count():,}")
print(f"Testing Records  : {testing_dataframe.count():,}")

print("\nTraining Dataset Schema")

training_dataframe.printSchema()


Training Records : 16,798,069
Testing Records  : 4,201,931

Training Dataset Schema
root
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: long (nullable = true)
 |-- isFlaggedFraud: long (nullable = true)
 |-- transactionTypeIndex: double (nullable = true)
 |-- assembledFeatures: vector (nullable = true)
 |-- features: vector (nullable = true)



#### Optimizing machine learning models

In [46]:
best_parameters = {}
trained_models = {}

binary_classification_evaluator = BinaryClassificationEvaluator(
    labelCol="isFraud",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

### LOGISTIC REGRESSION

In [13]:
def logistic_regression_objective(trial):

    logistic_regression_classifier = LogisticRegression(
        featuresCol="features",
        labelCol="isFraud",
        regParam=trial.suggest_float(
            "regParam",
            1e-5,
            1.0,
            log=True
        ),
        elasticNetParam=trial.suggest_float(
            "elasticNetParam",
            0.0,
            1.0
        ),
        maxIter=trial.suggest_int(
            "maxIter",
            50,
            200
        )
    )

    logistic_regression_model = logistic_regression_classifier.fit(
        training_dataframe
    )

    prediction_dataframe = logistic_regression_model.transform(
        testing_dataframe
    )

    score = binary_classification_evaluator.evaluate(
        prediction_dataframe
    )

    return score

In [14]:
logistic_regression_study = optuna.create_study(
    direction="maximize"
)

logistic_regression_study.optimize(
    logistic_regression_objective,
    n_trials=5
)

print("Best Parameters:")
print(logistic_regression_study.best_params)


[I 2026-06-28 08:57:40,045] A new study created in memory with name: no-name-938bce1e-9d47-4442-a0dc-025870f1bc77
[I 2026-06-28 08:58:11,788] Trial 0 finished with value: 0.0013351004573849499 and parameters: {'regParam': 0.0010786856655126363, 'elasticNetParam': 0.9164657514176534, 'maxIter': 153}. Best is trial 0 with value: 0.0013351004573849499.
[I 2026-06-28 08:58:31,909] Trial 1 finished with value: 0.0013351004573849499 and parameters: {'regParam': 0.02489828541713551, 'elasticNetParam': 0.08470204595538555, 'maxIter': 190}. Best is trial 0 with value: 0.0013351004573849499.
[I 2026-06-28 08:58:51,465] Trial 2 finished with value: 0.0013351004573849499 and parameters: {'regParam': 0.0014450028541642915, 'elasticNetParam': 0.5053759756878364, 'maxIter': 134}. Best is trial 0 with value: 0.0013351004573849499.
[I 2026-06-28 08:59:11,277] Trial 3 finished with value: 0.0013351004573849499 and parameters: {'regParam': 0.0012228974216611749, 'elasticNetParam': 0.9274388029946984, 'ma

Best Parameters:
{'regParam': 0.0010786856655126363, 'elasticNetParam': 0.9164657514176534, 'maxIter': 153}


In [47]:
best_parameters["Logistic Regression"] = logistic_regression_study.best_params

### DECISION TREE

In [16]:
def decision_tree_objective(trial):

    classifier = DecisionTreeClassifier(
        featuresCol="features",
        labelCol="isFraud",

        maxDepth=trial.suggest_int(
            "maxDepth",
            3,
            20
        ),

        maxBins=trial.suggest_int(
            "maxBins",
            16,
            64
        ),

        minInstancesPerNode=trial.suggest_int(
            "minInstancesPerNode",
            1,
            20
        )
    )

    model = classifier.fit(training_dataframe)

    predictions = model.transform(testing_dataframe)

    return binary_classification_evaluator.evaluate(predictions)


In [17]:
decision_tree_study = optuna.create_study(
    direction="maximize"
)

decision_tree_study.optimize(
    decision_tree_objective,
    n_trials=5
)

[I 2026-06-28 09:03:10,894] A new study created in memory with name: no-name-6035b56d-a936-47e8-8783-6f64420c6b4a
[I 2026-06-28 09:03:39,366] Trial 0 finished with value: 0.0013351004573849499 and parameters: {'maxDepth': 5, 'maxBins': 36, 'minInstancesPerNode': 5}. Best is trial 0 with value: 0.0013351004573849499.
[I 2026-06-28 09:04:11,474] Trial 1 finished with value: 0.0013252633428478913 and parameters: {'maxDepth': 9, 'maxBins': 64, 'minInstancesPerNode': 2}. Best is trial 0 with value: 0.0013351004573849499.
[I 2026-06-28 09:04:37,573] Trial 2 finished with value: 0.0013351004573849499 and parameters: {'maxDepth': 3, 'maxBins': 35, 'minInstancesPerNode': 17}. Best is trial 0 with value: 0.0013351004573849499.
[I 2026-06-28 09:05:02,873] Trial 3 finished with value: 0.0013351004573849499 and parameters: {'maxDepth': 4, 'maxBins': 44, 'minInstancesPerNode': 9}. Best is trial 0 with value: 0.0013351004573849499.
[I 2026-06-28 09:05:31,637] Trial 4 finished with value: 0.0013351004

In [18]:
print(decision_tree_study.best_params)

{'maxDepth': 5, 'maxBins': 36, 'minInstancesPerNode': 5}


In [48]:
best_parameters["Decision Tree"] = decision_tree_study.best_params

### Naive Bayes

In [20]:
# training_dataframe.cache()
# testing_dataframe.cache()

# train_sample = training_dataframe.sample(fraction=0.2, seed=42)

def nb_objective(trial):

    classifier = NaiveBayes(
        featuresCol="features",
        labelCol="isFraud",
        smoothing=trial.suggest_float("smoothing", 0.5, 2.0)
    )

    model = classifier.fit(training_dataframe)

    predictions = model.transform(testing_dataframe)

    return binary_classification_evaluator.evaluate(predictions)

In [21]:
nb_study = optuna.create_study(direction="maximize")

nb_study.optimize(
    nb_objective,
    n_trials=5  # keeping small for stability
)

[I 2026-06-28 09:05:31,652] A new study created in memory with name: no-name-d351393c-9d9e-4a46-acc9-2586d55b9731
[I 2026-06-28 09:06:07,290] Trial 0 finished with value: 0.0013231782995564568 and parameters: {'smoothing': 0.8138880630827166}. Best is trial 0 with value: 0.0013231782995564568.
[I 2026-06-28 09:06:35,541] Trial 1 finished with value: 0.0013232254398126263 and parameters: {'smoothing': 1.1536114082510942}. Best is trial 1 with value: 0.0013232254398126263.
[I 2026-06-28 09:07:03,993] Trial 2 finished with value: 0.0013232210625952834 and parameters: {'smoothing': 1.3128091207295316}. Best is trial 1 with value: 0.0013232254398126263.
[I 2026-06-28 09:07:31,171] Trial 3 finished with value: 0.0013232345178520544 and parameters: {'smoothing': 1.2197698013281952}. Best is trial 3 with value: 0.0013232345178520544.
[I 2026-06-28 09:07:59,009] Trial 4 finished with value: 0.00132323979009409 and parameters: {'smoothing': 1.987004458770097}. Best is trial 4 with value: 0.00132

In [22]:
print("Best Params:", nb_study.best_params)

Best Params: {'smoothing': 1.987004458770097}


In [49]:
best_parameters["Naive Bayes"] = decision_tree_study.best_params

## Random Forest

In [50]:

# train_sample = training_dataframe.sample(fraction=0.3, seed=42)
def random_forest_objective(trial):

    classifier = RandomForestClassifier(
        featuresCol="features",
        labelCol="isFraud",

        numTrees=trial.suggest_int(
            "numTrees",
            10,
            50
        ),

        maxDepth=trial.suggest_int(
            "maxDepth",
            3,
            5
        ),

        maxBins=trial.suggest_int(
            "maxBins",
            16,
            32
        )
    )

    model = classifier.fit(training_dataframe)

    predictions = model.transform(testing_dataframe)

    return binary_classification_evaluator.evaluate(predictions)

In [ ]:
random_forest_study = optuna.create_study(
    direction="maximize"
)

random_forest_study.optimize(
    random_forest_objective,
    n_trials=5
)

[I 2026-06-28 09:09:26,463] A new study created in memory with name: no-name-bf5e616f-2b83-4628-9010-109b0a0813b1
[I 2026-06-28 09:12:52,459] Trial 0 finished with value: 0.0013351004573849499 and parameters: {'numTrees': 48, 'maxDepth': 4, 'maxBins': 16}. Best is trial 0 with value: 0.0013351004573849499.
[I 2026-06-28 09:15:46,265] Trial 1 finished with value: 0.001543224250293978 and parameters: {'numTrees': 32, 'maxDepth': 5, 'maxBins': 21}. Best is trial 1 with value: 0.001543224250293978.
[I 2026-06-28 09:17:08,369] Trial 2 finished with value: 0.0013351004573849499 and parameters: {'numTrees': 23, 'maxDepth': 3, 'maxBins': 17}. Best is trial 1 with value: 0.001543224250293978.
[I 2026-06-28 09:18:05,871] Trial 3 finished with value: 0.0013351004573849499 and parameters: {'numTrees': 16, 'maxDepth': 3, 'maxBins': 32}. Best is trial 1 with value: 0.001543224250293978.


In [11]:
print(random_forest_study.best_params)

{'numTrees': 15, 'maxDepth': 4, 'maxBins': 18}


In [51]:
best_parameters["Random Forest"] = random_forest_study.best_params

### Best Parameters

In [53]:
for model_name, parameters in best_parameters.items():

    print("=" * 60)
    print(model_name)

    for parameter, value in parameters.items():
        print(f"{parameter}: {value}")

Logistic Regression
regParam: 0.0010786856655126363
elasticNetParam: 0.9164657514176534
maxIter: 153
Decision Tree
maxDepth: 5
maxBins: 36
minInstancesPerNode: 5
Naive Bayes
smoothing: 1.987004458770097
Random Forest
numTrees: 32
maxDepth: 5
maxBins: 21


In [54]:
#saving best parameter sets
import json
import os

os.makedirs("results", exist_ok=True)

with open("results/best_parameters.json", "w") as file:
    json.dump(best_parameters, file, indent=4)

### Evaluations

In [55]:

trained_models = {}
model_summary = []

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="isFraud",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="isFraud",
    predictionCol="prediction",
    metricName="accuracy"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="isFraud",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="isFraud",
    predictionCol="prediction",
    metricName="weightedRecall"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="isFraud",
    predictionCol="prediction",
    metricName="f1"
)

In [17]:
with open("results/best_parameters.json", "r") as file:
    best_parameters = json.load(file)

print(best_parameters)

{'Logistic Regression': {'regParam': 0.0010786856655126363, 'elasticNetParam': 0.9164657514176534, 'maxIter': 153}, 'Decision Tree': {'maxDepth': 5, 'maxBins': 36, 'minInstancesPerNode': 5}, 'Naive Bayes': {'smoothing': 1.987004458770097}, 'Random Forest': {'numTrees': 32, 'maxDepth': 5, 'maxBins': 21}}


In [57]:
lr_params = best_parameters["Logistic Regression"]
dt_params = best_parameters["Decision Tree"]
nb_params = best_parameters["Naive Bayes"]
rf_params = best_parameters["Random Forest"]

In [58]:
model_summary = []

def evaluate_model(name, classifier, params):

    print("\n" + "=" * 70)
    print(f"Training {name}")
    print("=" * 70)

    start = time.time()

    model = classifier.fit(training_dataframe)

    training_time = time.time() - start

    predictions = model.transform(testing_dataframe)

    accuracy = accuracy_evaluator.evaluate(predictions)
    precision = precision_evaluator.evaluate(predictions)
    recall = recall_evaluator.evaluate(predictions)
    f1 = f1_evaluator.evaluate(predictions)
    auc = auc_evaluator.evaluate(predictions)

    print(f"Training Time : {training_time:.2f} seconds")
    print(f"Accuracy      : {accuracy:.4f}")
    print(f"Precision     : {precision:.4f}")
    print(f"Recall        : {recall:.4f}")
    print(f"F1 Score      : {f1:.4f}")
    print(f"AUC-ROC       : {auc:.4f}")

    model_summary.append({
        "Algorithm": name,
        "Hyperparameters": ", ".join(
            [f"{k}={v}" for k, v in params.items()]
        ),
        "Training Time (s)": round(training_time, 2),
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-Score": round(f1, 4),
        "AUC-ROC": round(auc, 4)
    })

In [59]:
# ---------------------------------------------------
# Logistic Regression
# ---------------------------------------------------

lr_params = best_parameters["Logistic Regression"]

lr = LogisticRegression(
    featuresCol="features",
    labelCol="isFraud",
    **lr_params
)

evaluate_model(
    "Logistic Regression",
    lr,
    lr_params
)

# ---------------------------------------------------
# Decision Tree
# ---------------------------------------------------

dt_params = best_parameters["Decision Tree"]

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="isFraud",
    **dt_params
)

evaluate_model(
    "Decision Tree",
    dt,
    dt_params
)

# ---------------------------------------------------
# Naive Bayes
# ---------------------------------------------------

nb_params = best_parameters["Naive Bayes"]

nb = NaiveBayes(
    featuresCol="features",
    labelCol="isFraud",
    **nb_params
)

evaluate_model(
    "Naive Bayes",
    nb,
    nb_params
)

# ---------------------------------------------------
# Random Forest
# ---------------------------------------------------

rf_params = best_parameters["Random Forest"]

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="isFraud",
    **rf_params
)

evaluate_model(
    "Random Forest",
    rf,
    rf_params
)



Training Logistic Regression
Training Time : 101.56 seconds
Accuracy      : 0.9987
Precision     : 0.9974
Recall        : 0.9987
F1 Score      : 0.9981
AUC-ROC       : 0.5000

Training Decision Tree
Training Time : 238.48 seconds
Accuracy      : 0.9987
Precision     : 0.9974
Recall        : 0.9987
F1 Score      : 0.9981
AUC-ROC       : 0.5000

Training Naive Bayes
Training Time : 37.10 seconds
Accuracy      : 0.9987
Precision     : 0.9974
Recall        : 0.9987
F1 Score      : 0.9981
AUC-ROC       : 0.5030

Training Random Forest
Training Time : 809.66 seconds
Accuracy      : 0.9987
Precision     : 0.9974
Recall        : 0.9987
F1 Score      : 0.9981
AUC-ROC       : 0.5031


In [60]:
summary_df = pd.DataFrame(model_summary)

print("\n")
print("=" * 100)
print("MODEL SUMMARY")
print("=" * 100)
print(summary_df)

os.makedirs("results", exist_ok=True)

summary_df.to_csv(
    "results/model_summary.csv",
    index=False
)

print("\nSaved to results/model_summary.csv")



MODEL SUMMARY
             Algorithm                                    Hyperparameters  \
0  Logistic Regression  regParam=0.0010786856655126363, elasticNetPara...   
1        Decision Tree      maxDepth=5, maxBins=36, minInstancesPerNode=5   
2          Naive Bayes                        smoothing=1.987004458770097   
3        Random Forest                numTrees=32, maxDepth=5, maxBins=21   

   Training Time (s)  Accuracy  Precision  Recall  F1-Score  AUC-ROC  
0             101.56    0.9987     0.9974  0.9987    0.9981   0.5000  
1             238.48    0.9987     0.9974  0.9987    0.9981   0.5000  
2              37.10    0.9987     0.9974  0.9987    0.9981   0.5030  
3             809.66    0.9987     0.9974  0.9987    0.9981   0.5031  

Saved to results/model_summary.csv


In [20]:
rf_params = best_parameters["Random Forest"]

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="isFraud",
    **rf_params
)
# Get feature importance (example: Random Forest)
rf_model = rf.fit(training_dataframe)

importances = rf_model.featureImportances.toArray()



In [22]:
feature_cols = [
    "type_index",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "isFlaggedFraud"
]

# Create DataFrame
feature_importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

# Save for Tableau
feature_importance_df.to_csv("feature_importance.csv", index=False)

print(feature_importance_df)

          Feature  Importance
2   oldbalanceOrg    0.307950
3  newbalanceOrig    0.212803
1          amount    0.199314
0      type_index    0.161888
6  isFlaggedFraud    0.076952
4  oldbalanceDest    0.041092
5  newbalanceDest    0.000000
